# One customer, end to end

This notebook follows **a single person** through the identity-resolution pipeline,
in the order the pipeline actually works:

| | |
|---|---|
| 1 | the raw records, scattered across source systems and disagreeing with each other |
| 2 | what normalisation repairs, and what it cannot |
| 3 | the vector that gets built from each record |
| 4 | the two ways the pipeline looks for candidates — meaning, and exact keys |
| 5 | how those two lists are fused and ranked into one score and one tier |
| 6 | what the model was asked to decide, and what it decided |
| 7 | the merged golden record, and which source won each field |

Nothing here builds anything. Run `./run.sh` first, then work down this notebook.
Every cell runs top to bottom with no edits: the project, the dataset and the
person are all read from `config.env` or chosen by query, never typed in.

> If a cell returns no rows, the pipeline stage above it has not been run yet.

## Setup

We talk to BigQuery through the `bq` command-line tool, so this notebook needs no
Python client libraries — just a `gcloud` install you are already logged in to.

`config.env` is written by `setup.sh` and is the only place a project name lives.
Nothing below hardcodes one.

In [ ]:
import json, os, re, subprocess, sys, textwrap
from pathlib import Path

# --- find config.env -----------------------------------------------------
# Look for an explicit override first, then walk up from wherever this
# notebook was opened. Works from demo/, from the repo root, or from a
# copy of this file somewhere else in the tree.
def _find_config():
    override = os.environ.get("CDP_CONFIG_ENV")
    if override and Path(override).is_file():
        return Path(override)
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        for cand in (d / "config.env", d / "demo" / "config.env"):
            if cand.is_file():
                return cand
    raise SystemExit("No config.env found — run ./setup.sh first.")

CONFIG = _find_config()
CFG = {}
for _line in CONFIG.read_text().splitlines():
    _m = re.match(r'^([A-Z0-9_]+)="(.*)"$', _line.strip())
    if _m:
        CFG[_m.group(1)] = _m.group(2)

PROJECT  = CFG["CDP_PROJECT"]
DATASET  = CFG.get("CDP_DS") or CFG["CDP_DATASET_PREFIX"]
LOCATION = CFG.get("CDP_LOCATION", "US")
DS       = f"{PROJECT}.{DATASET}"          # substituted into SQL as {ds}

# Some sandboxes keep a writable copy of the gcloud config here. Only used
# when the environment has not already set one.
_alt_gcloud = Path("/tmp/gcloudcfg")
if "CLOUDSDK_CONFIG" not in os.environ and _alt_gcloud.is_dir():
    os.environ["CLOUDSDK_CONFIG"] = str(_alt_gcloud)

# --- pandas is optional --------------------------------------------------
try:
    import pandas as pd
except ImportError:
    pd = None


def q(sql, max_rows=50):
    """Run a query and return a list of dicts. Values come back as strings."""
    sql = sql.replace("{ds}", DS)
    p = subprocess.run(
        ["bq", "query", f"--project_id={PROJECT}", f"--location={LOCATION}",
         "--use_legacy_sql=false", "--format=json", "--quiet",
         "--max_rows", str(max_rows), sql],
        capture_output=True, text=True, timeout=600)
    if p.returncode != 0:
        # bq reports query errors on stdout, not stderr.
        raise RuntimeError((p.stdout + p.stderr).strip()[-2000:])
    out = p.stdout
    start = out.find("[")
    return json.loads(out[start:]) if start >= 0 else []


def lit(value):
    """Quote a Python string as a BigQuery string literal."""
    s = str(value).replace("\\", "\\\\").replace("'", "\\'")
    return "'" + s.replace("\n", " ") + "'"


def show(rows, cols=None, width=40, empty="(no rows)"):
    """Render rows as a table — a DataFrame if pandas is here, text if not."""
    if not rows:
        print(empty)
        return
    # Keep the requested order, but never ask for a column the query did not
    # return — bq returns its JSON keys alphabetically, so an explicit list is
    # the only way to control column order.
    available = list(rows[0].keys())
    cols = [c for c in (cols or available) if c in available] or available
    if pd is not None:
        try:
            from IPython.display import display
            display(pd.DataFrame([{c: r.get(c) for c in cols} for r in rows],
                                 columns=cols))
            return
        except Exception:
            pass
    def cell(v):
        s = "" if v is None else str(v)
        return s if len(s) <= width else s[: width - 1] + "…"
    w = {c: max([len(c)] + [len(cell(r.get(c))) for r in rows]) for c in cols}
    print("  ".join(c.ljust(w[c]) for c in cols))
    print("  ".join("-" * w[c] for c in cols))
    for r in rows:
        print("  ".join(cell(r.get(c)).ljust(w[c]) for c in cols))


def show_record(row, cols=None, wrap=88):
    """Render one wide row vertically — easier to read than 30 columns."""
    if not row:
        print("(no row)")
        return
    available = list(row.keys())
    cols = [c for c in (cols or available) if c in available] or available
    pad = max(len(c) for c in cols)
    for c in cols:
        v = "" if row.get(c) is None else str(row[c])
        lines = textwrap.wrap(v, wrap) or [""]
        print(f"{c.rjust(pad)} : {lines[0]}")
        for extra in lines[1:]:
            print(f"{' ' * pad}   {extra}")


def run_visible(sql, max_rows=50):
    """Print the SQL, then run it. Used where the SQL is the point."""
    print(sql.replace("{ds}", DS).strip())
    print("-" * 78)
    return q(sql, max_rows=max_rows)


print(f"config     {CONFIG}")
print(f"project    {PROJECT}")
print(f"dataset    {DATASET}   (location {LOCATION})")
print(f"pandas     {'yes' if pd else 'no — falling back to printed tables'}")

_corpus = q("""
SELECT (SELECT COUNT(*) FROM `{ds}.party_records`)                      AS source_records,
       (SELECT COUNT(DISTINCT source_system) FROM `{ds}.party_records`) AS source_systems,
       (SELECT COUNT(DISTINCT person_id) FROM `{ds}.person_assignment`) AS resolved_people
""")[0]
print(f"corpus     {_corpus['source_records']} records from "
      f"{_corpus['source_systems']} source systems, resolved into "
      f"{_corpus['resolved_people']} people")

## 1 · Pick the customer

We want one person whose story is worth telling: several records, spread over at
least three different systems, and at least one of those records coming from an
unstructured source — a call transcript or a support ticket, where the identity had
to be *read out of free text* rather than looked up in a column.

The corpus is regenerated from scratch on every run, so the person is **chosen by
query**, not named here. If the strict criteria find nobody, the selection relaxes
step by step rather than failing.

In [ ]:
SUBJECT_SQL = """
WITH members AS (
  SELECT person_id, record_id FROM `{ds}.person_assignment`
),
stats AS (
  SELECT m.person_id,
         COUNT(*)                                        AS n_records,
         COUNT(DISTINCT pr.source_system)                AS n_sources,
         STRING_AGG(DISTINCT pr.source_system ORDER BY pr.source_system) AS sources,
         COUNTIF(pr.source_system IN ('CALL','SUPPORT'))  AS n_unstructured
  FROM members AS m
  JOIN `{ds}.party_records` AS pr USING (record_id)
  GROUP BY m.person_id
),
-- Did any pair inside this cluster have to go to the adjudicator?
grey AS (
  SELECT a.person_id, COUNT(*) AS n_grey
  FROM `{ds}.pair_tiers` AS t
  JOIN members AS a ON a.record_id = t.record_id_a
  JOIN members AS b ON b.record_id = t.record_id_b AND b.person_id = a.person_id
  WHERE t.tier = 'GREY_ZONE'
  GROUP BY a.person_id
)
SELECT s.person_id, s.n_records, s.n_sources, s.sources, s.n_unstructured,
       IFNULL(g.n_grey, 0) AS grey_zone_pairs,
       gp.full_name, gp.city, gp.postcode
FROM stats AS s
LEFT JOIN grey AS g USING (person_id)
LEFT JOIN `{ds}.golden_person` AS gp USING (person_id)
WHERE {filter}
-- Deterministic: prefer a cluster the adjudicator actually had to rule on,
-- then breadth of sources, then size. person_id breaks any remaining tie.
ORDER BY IF(IFNULL(g.n_grey, 0) > 0, 1, 0) DESC,
         s.n_sources DESC, s.n_unstructured DESC, s.n_records DESC, s.person_id
LIMIT 1
"""

CRITERIA = [
    ("3+ systems, 4-8 records, at least one unstructured source",
     "s.n_sources >= 3 AND s.n_unstructured >= 1 AND s.n_records BETWEEN 4 AND 8"),
    ("3+ systems, 4-8 records",
     "s.n_sources >= 3 AND s.n_records BETWEEN 4 AND 8"),
    ("2+ systems, more than one record",
     "s.n_sources >= 2 AND s.n_records > 1"),
]

SUBJECT = None
for _label, _pred in CRITERIA:
    _rows = q(SUBJECT_SQL.replace("{filter}", _pred), max_rows=1)
    if _rows:
        SUBJECT, CRITERIA_USED = _rows[0], _label
        break
if SUBJECT is None:
    raise SystemExit("No multi-record clusters found — has the pipeline been run?")

PERSON_ID = SUBJECT["person_id"]

print(f"selected on : {CRITERIA_USED}\n")
show_record(SUBJECT, cols=["person_id", "full_name", "city", "postcode",
                           "n_records", "n_sources", "sources",
                           "n_unstructured", "grey_zone_pairs"])
print(f"\nEverything below follows {SUBJECT.get('full_name') or PERSON_ID} "
      f"({PERSON_ID}) and nobody else.")

## 2 · The raw records

Here is the customer as the business actually holds them today: one row per system
that has ever seen them. Read down the `raw_name` column first.

Nothing has been cleaned yet. Names are abbreviated, misspelled or simply different;
postcodes are truncated or mistyped; phone numbers are written in whatever format the
system that captured them happened to use; and most systems hold only a fraction of
the fields. Two records can be the same human and share almost no characters.

The counts printed under the table are computed from the rows above them, so they
describe this customer rather than a generalisation.

`source_trust` is the operator's own ranking of how reliable each system is — it
matters later, when the pipeline has to choose between two values that disagree.

In [ ]:
RAW_SQL = """
SELECT pr.record_id, pr.source_system, pr.source_trust, pr.source_natural_key,
       pr.raw_name, pr.raw_address, pr.raw_city, pr.raw_postcode,
       pr.raw_email, pr.raw_phone, pr.raw_dob, pr.account_number
FROM `{ds}.person_assignment` AS pa
JOIN `{ds}.party_records`     AS pr USING (record_id)
WHERE pa.person_id = @pid
ORDER BY pr.source_trust DESC, pr.source_system, pr.record_id
LIMIT 30
"""
RAW = q(RAW_SQL.replace("@pid", lit(PERSON_ID)))
show(RAW, cols=["record_id", "source_system", "source_trust", "raw_name",
                "raw_address", "raw_city", "raw_postcode", "raw_email",
                "raw_phone", "raw_dob", "account_number"], width=32)

# Counted, not asserted: how much do these records actually disagree?
def _distinct(rows, col):
    return sorted({r[col] for r in rows if r.get(col)})

for _field in ("raw_name", "raw_email", "raw_phone", "raw_postcode",
               "raw_dob", "account_number"):
    _vals = _distinct(RAW, _field)
    _missing = sum(1 for r in RAW if not r.get(_field))
    print(f"{_field:<16} {len(_vals)} distinct value(s), "
          f"{_missing} of {len(RAW)} records leave it blank"
          + (f"  ->  {_vals}" if 0 < len(_vals) <= 6 else ""))

## 3 · Normalisation — what cleaning can fix, and what it cannot

Stage 20 puts every record through the same small set of functions: fold accents and
punctuation out of names, force phone numbers to international `+61…` form, strip
the noise out of postcodes, lowercase and de-alias emails.

This is the cheap, deterministic half of the problem, and it genuinely helps —
`0419 322 737` and `+61419322737` become the same string, so a plain equality test
now finds them.

It is also where the limit shows. Normalisation repairs *formatting*. It cannot
repair *content*: a misspelled name stays misspelled, a truncated postcode stays
truncated, and a blank field stays blank. Everything that survives this cell is the
hard part, and is why the rest of the pipeline exists.

`identity_strength` counts how many strong identifiers (email, phone, date of birth,
account number) a record actually carries — 0 means there is almost nothing here to
match on.

In [ ]:
NORM_SQL = """
SELECT pr.record_id, pr.source_system,
       pr.raw_name,     pr.name_norm,
       pr.raw_postcode, pr.postcode_norm,
       pr.raw_phone,    pr.phone_e164,
       pr.raw_email,    pr.email_norm,
       pr.dob, pr.account_number, pr.identity_strength
FROM `{ds}.person_assignment` AS pa
JOIN `{ds}.party_records`     AS pr USING (record_id)
WHERE pa.person_id = @pid
ORDER BY pr.identity_strength DESC, pr.record_id
LIMIT 30
"""
NORM = q(NORM_SQL.replace("@pid", lit(PERSON_ID)))
show(NORM, cols=["record_id", "source_system", "raw_name", "name_norm",
                 "raw_phone", "phone_e164", "raw_postcode", "postcode_norm",
                 "raw_email", "email_norm", "identity_strength"])

print("\nwhat the normalisers changed on this customer")
for _raw, _norm, _label in (("raw_name", "name_norm", "name"),
                            ("raw_phone", "phone_e164", "phone"),
                            ("raw_postcode", "postcode_norm", "postcode"),
                            ("raw_email", "email_norm", "email")):
    _rewritten = sum(1 for r in NORM
                     if r.get(_raw) and r.get(_norm) and r[_raw] != r[_norm])
    _dropped = sum(1 for r in NORM if r.get(_raw) and not r.get(_norm))
    _blank = sum(1 for r in NORM if not r.get(_raw))
    print(f"  {_label:<9} rewritten {_rewritten}   "
          f"rejected as unusable {_dropped}   already blank {_blank}")

print("\nstill different after cleaning:")
print("  names    ", sorted({r["name_norm"] for r in NORM if r.get("name_norm")}))
print("  emails   ", sorted({r["email_norm"] for r in NORM if r.get("email_norm")}))
print("  accounts ", sorted({r["account_number"] for r in NORM if r.get("account_number")}))

## 4 · The vector

Each record is condensed into one `match_key` string — name, address, postcode,
email, phone, account number, whatever that record happens to have — and that
string is embedded into a vector by a generated column on `party_search`.

A vector is just a long list of numbers that positions the record in a space where
"close together" means "means something similar". It is what lets the pipeline see
that `DYYLAN KITH` and `DYLAN KEITH` are the same idea, when no exact-match rule
ever could.

We take the customer's strongest record as the probe for the two searches that
follow. Only the first few components are printed — there are thousands.

In [ ]:
PROBE_RECORD = NORM[0]["record_id"]          # highest identity_strength, tie-broken by id

VECTOR_SQL = """
SELECT ps.record_id, ps.source_system, ps.match_key,
       ARRAY_LENGTH(ps.match_embedding.result)          AS dimensions,
       IFNULL(NULLIF(ps.match_embedding.status, ''), 'ok') AS embed_status,
       (SELECT ROUND(SQRT(SUM(v * v)), 4)
        FROM UNNEST(ps.match_embedding.result) AS v)    AS vector_length,
       (SELECT STRING_AGG(FORMAT('%+.4f', v), '  ' ORDER BY o)
        FROM UNNEST(ps.match_embedding.result) AS v WITH OFFSET AS o
        WHERE o < 8)                                    AS first_8_components
FROM `{ds}.party_search` AS ps
WHERE ps.record_id = @rid
"""
VEC = q(VECTOR_SQL.replace("@rid", lit(PROBE_RECORD)))[0]
PROBE = VEC["match_key"]

print("the string that was embedded")
print(f"  {PROBE}\n")
print(f"dimensions      {VEC['dimensions']}")
print(f"embed status    {VEC['embed_status']}")
print(f"vector length   {VEC['vector_length']}   (normalised, so cosine is a pure angle)")
print(f"first 8 of {VEC['dimensions']}  {VEC['first_8_components']}  …")
print(f"\nmodel           {CFG.get('CDP_EMBEDDING_ENDPOINT')}")

## 5 · Two ways to look for the same person

The pipeline searches for candidates twice, because the two methods fail in
completely different ways.

**Semantic search** compares vectors. It copes with misspellings, nicknames and
missing fields, because it is matching meaning rather than characters. It is also
blind to identifiers: two *different* account numbers look almost identical to an
embedding, and it will happily return a stranger who lives on a similar-sounding
street. Expect to see exactly that in the results below.

**Keyword / BM25 search** compares the actual tokens. It is exact and cheap and it
never confuses two account numbers — but it finds nothing at all when the name is
misspelled or the shared field is blank.

Run below: the same probe string, both ways.

In [ ]:
SEMANTIC_SQL = """
-- Semantic leg. Same body as the `tf_lookup_vector` table function.
SELECT base.record_id, base.source_system, base.match_key,
       ROUND(distance, 4) AS distance          -- 0.0 = identical
FROM AI.SEARCH(
       TABLE `{ds}.party_search`,
       'match_key',
       @probe,
       top_k => 10,
       mode  => 'VECTOR')
ORDER BY distance
"""
SEM = run_visible(SEMANTIC_SQL.replace("@probe", lit(PROBE)))
show(SEM, width=60)

### The keyword leg

Same index, different mode. `HYBRID` runs BM25 over the stored `match_key` text and
the vector search together, and fuses the two rankings inside BigQuery.

One practical note, visible in the SQL: the probe is inlined as a literal rather than
passed to the `tf_lookup_hybrid` table function. AI.SEARCH in hybrid mode requires the
query string to be a constant, and a table-function parameter is not one — calling the
function fails with *"query_value argument of AI.SEARCH must be a non-null STRING for
hybrid search"*. The body is otherwise identical to the shipped function.

In [ ]:
HYBRID_SQL = """
-- Keyword + vector in one call, fused by the index (BM25 over match_key).
SELECT base.record_id, base.source_system, base.match_key,
       ROUND(distance, 4) AS distance
FROM AI.SEARCH(
       TABLE `{ds}.party_search`,
       'match_key',
       @probe,
       top_k => 10,
       mode  => 'HYBRID')
ORDER BY distance
"""
HYB = run_visible(HYBRID_SQL.replace("@probe", lit(PROBE)))
show(HYB, cols=["record_id", "source_system", "distance", "match_key"], width=58)

_sem_ids = [r["record_id"] for r in SEM]
_hyb_ids = [r["record_id"] for r in HYB]
print(f"\nprobe record        {PROBE_RECORD}")
print(f"returned by both    {sorted(set(_sem_ids) & set(_hyb_ids))}")
print(f"semantic leg only   {sorted(set(_sem_ids) - set(_hyb_ids))}")
print(f"keyword leg only    {sorted(set(_hyb_ids) - set(_sem_ids))}")

### Which leg found which candidate

The live searches above are one probe. The pipeline does this for every record, and
for each surviving pair it records whether the *deterministic key* leg found it
(shared email, phone, account number, postcode + soundex), whether the *semantic*
leg found it, or both — and at what rank in each list.

That is the column to read below: `retrieved_by`. A pair marked `SEMANTIC_ONLY` is
one that no deterministic rule would ever have proposed — the records share no key
at all. A pair marked `LEXICAL_ONLY` is one the embedding missed.

The counts below cover **every** pair touching this customer, not just the top of
the list, so the split is the real one.

In [ ]:
LEG_COUNTS_SQL = """
WITH members AS (
  SELECT record_id, person_id FROM `{ds}.person_assignment`
),
mine AS (
  SELECT t.*
  FROM `{ds}.pair_tiers` AS t
  JOIN members AS ma ON ma.record_id = t.record_id_a
  JOIN members AS mb ON mb.record_id = t.record_id_b
  WHERE ma.person_id = @pid OR mb.person_id = @pid
)
SELECT retrieved_by,
       COUNT(*)                      AS pairs,
       COUNTIF(tier = 'AUTO_MATCH')  AS became_auto_match,
       COUNTIF(tier = 'GREY_ZONE')   AS became_grey_zone,
       COUNTIF(tier = 'REJECT')      AS became_reject
FROM mine
GROUP BY retrieved_by
ORDER BY pairs DESC
"""
LEG_COUNTS = q(LEG_COUNTS_SQL.replace("@pid", lit(PERSON_ID)))
print("every pair touching this customer, by the leg that found it")
show(LEG_COUNTS, cols=["retrieved_by", "pairs", "became_auto_match",
                       "became_grey_zone", "became_reject"])

# Examples from each leg, so the single-leg pairs are named rather than counted.
LEG_EXAMPLES_SQL = """
WITH members AS (
  SELECT record_id, person_id FROM `{ds}.person_assignment`
),
mine AS (
  SELECT t.*, (ma.person_id = mb.person_id) AS both_in_our_cluster
  FROM `{ds}.pair_tiers` AS t
  JOIN members AS ma ON ma.record_id = t.record_id_a
  JOIN members AS mb ON mb.record_id = t.record_id_b
  WHERE ma.person_id = @pid OR mb.person_id = @pid
)
SELECT record_id_a, record_id_b, retrieved_by, both_in_our_cluster,
       matched_keys                 AS keys_the_lexical_leg_matched_on,
       rank_lexical, rank_semantic,
       ROUND(similarity, 3)         AS cosine_similarity,
       ROUND(rrf_score, 5)          AS rrf_score,
       tier
FROM mine
QUALIFY ROW_NUMBER() OVER (PARTITION BY retrieved_by
                           ORDER BY rrf_score DESC, record_id_a) <= 3
ORDER BY retrieved_by, rrf_score DESC
LIMIT 20
"""
LEGS = q(LEG_EXAMPLES_SQL.replace("@pid", lit(PERSON_ID)))
print("\nup to three examples from each leg")
show(LEGS, cols=["retrieved_by", "record_id_a", "record_id_b",
                 "both_in_our_cluster", "keys_the_lexical_leg_matched_on",
                 "rank_lexical", "rank_semantic", "cosine_similarity",
                 "rrf_score", "tier"], width=30)

_legs_present = {r["retrieved_by"] for r in LEG_COUNTS}
for _leg, _meaning in (("SEMANTIC_ONLY", "share no deterministic key at all — "
                                         "only the embedding connected them"),
                       ("LEXICAL_ONLY", "share an exact key, but the embedding "
                                        "did not rank them near each other")):
    _hits = [r for r in LEGS if r["retrieved_by"] == _leg]
    if _hits:
        print(f"\n{_leg}: pairs that {_meaning}")
        for r in _hits:
            print(f"  {r['record_id_a']} ~ {r['record_id_b']}  "
                  f"cosine {r['cosine_similarity']}  -> {r['tier']}")
    else:
        print(f"\n{_leg}: none for this customer in this run "
              f"(v_retrieval_legs has the corpus-wide split).")

## 6 · Ranking — turning two lists into one decision

Now the two rankings have to be combined. The problem: a BM25 score and a cosine
distance have no common unit, so they cannot simply be averaged.

**Reciprocal rank fusion** sidesteps that by throwing the scores away and keeping only
the positions: each leg contributes `1 / (60 + its rank)`. A pair that both legs put
near the top wins; a pair that spikes in one leg and is absent from the other does
not. That is `rrf_score`.

Separately, a **rule score** adds up the hard evidence — same account number, same
email, same phone, same date of birth, name similarity — and subtracts for outright
contradictions. That is `rule_score`.

`combined_score` fuses those two, and the tier is then decided against thresholds
from `config.env`, printed below so they are not a mystery:

- at or above `tau_hi` with nothing contradicting → **AUTO_MATCH**, merged with no model call
- clearly below → **REJECT**
- everything in between → **GREY_ZONE**, and only these cost money

`tier_reason` is the plain-English version of whichever rule fired.

In [ ]:
print(f"tau_hi (auto-match at or above) : {CFG.get('CDP_TAU_HI')}")
print(f"tau_lo (grey zone down to)      : {CFG.get('CDP_TAU_LO')}")
print(f"single-signal floor             : {CFG.get('CDP_TAU_SINGLE_SIGNAL')}")
print(f"low-identity floor              : {CFG.get('CDP_TAU_LOW_IDENTITY')}\n")

RANK_SQL = """
WITH members AS (
  SELECT record_id, person_id FROM `{ds}.person_assignment`
)
SELECT t.record_id_a, t.record_id_b,
       (ma.person_id = mb.person_id)    AS ended_up_same_person,
       t.retrieved_by,
       t.strong_signals,                            -- identifiers that agree
       ROUND(t.rule_score, 3)           AS rule_score,
       ROUND(t.similarity, 3)           AS semantic_similarity,
       ROUND(t.rrf_score, 5)            AS rrf_score,
       ROUND(t.combined_score, 3)       AS combined_score,
       t.forename_conflict,
       t.tier, t.tier_reason
FROM `{ds}.pair_tiers` AS t
JOIN members AS ma ON ma.record_id = t.record_id_a
JOIN members AS mb ON mb.record_id = t.record_id_b
WHERE ma.person_id = @pid OR mb.person_id = @pid
ORDER BY t.combined_score DESC
LIMIT 20
"""
RANKED = q(RANK_SQL.replace("@pid", lit(PERSON_ID)))
show(RANKED, cols=["record_id_a", "record_id_b", "ended_up_same_person",
                   "retrieved_by", "strong_signals", "rule_score",
                   "semantic_similarity", "rrf_score", "combined_score", "tier"],
     width=22)

print("\nwhy each pair landed where it did")
for r in RANKED:
    print(f"  {r['record_id_a']:<15} ~ {r['record_id_b']:<15} "
          f"{r['combined_score']:>6}  {r['tier']:<11} {r['tier_reason']}")

_tiers = {}
for r in RANKED:
    _tiers[r["tier"]] = _tiers.get(r["tier"], 0) + 1
print("\ntier counts for this customer's pairs:", _tiers)

## 7 · Adjudication — the only pairs that cost money

Grey-zone pairs are the ones where the evidence genuinely points both ways, and they
are the only ones sent to a model. The prompt carries structured comparison features,
not raw records — the model is asked to weigh stated evidence, not to go fishing.

Every verdict is stamped with the model and prompt version that produced it, and the
ledger is append-only, so any decision can be reproduced later. Read
`decisive_evidence` and `contradiction`: they are the model's own account of what
settled it.

Note that a verdict is not automatically a merge. `UNCERTAIN`, or a `MATCH` below the
acceptance confidence, is parked for a human rather than acted on.

In [ ]:
ADJ_SQL = """
WITH members AS (
  SELECT record_id, person_id FROM `{ds}.person_assignment`
)
SELECT j.record_id_a, j.record_id_b, t.tier,
       ROUND(t.combined_score, 3) AS combined_score,
       j.verdict, ROUND(j.confidence, 2) AS confidence,
       j.decisive_evidence, j.contradiction, j.rationale,
       j.injection_detected, j.model, j.prompt_version, j.adjudicated_at
FROM `{ds}.v_adjudications_current` AS j
JOIN `{ds}.pair_tiers` AS t
  ON t.record_id_a = j.record_id_a AND t.record_id_b = j.record_id_b
JOIN members AS ma ON ma.record_id = j.record_id_a
JOIN members AS mb ON mb.record_id = j.record_id_b
WHERE ma.person_id = @pid OR mb.person_id = @pid
ORDER BY (ma.person_id = mb.person_id) DESC, j.confidence DESC
LIMIT 6
"""
ADJ = q(ADJ_SQL.replace("@pid", lit(PERSON_ID)))

if not ADJ:
    print(f"No pair touching {PERSON_ID} reached the grey zone in this run, so the\n"
          f"adjudicator was never asked about this customer. Every merge here was\n"
          f"settled by the rules alone. Showing an illustrative adjudicated pair\n"
          f"from elsewhere in the corpus instead:\n")
    ADJ = q("""
      SELECT j.record_id_a, j.record_id_b, t.tier,
             ROUND(t.combined_score, 3) AS combined_score,
             j.verdict, ROUND(j.confidence, 2) AS confidence,
             j.decisive_evidence, j.contradiction, j.rationale,
             j.injection_detected, j.model, j.prompt_version, j.adjudicated_at
      FROM `{ds}.v_adjudications_current` AS j
      JOIN `{ds}.pair_tiers` AS t
        ON t.record_id_a = j.record_id_a AND t.record_id_b = j.record_id_b
      WHERE j.rationale IS NOT NULL
      ORDER BY j.confidence DESC, j.record_id_a
      LIMIT 2
    """, max_rows=2)
else:
    print(f"{len(ADJ)} adjudicated pair(s) involve this customer "
          f"(model {ADJ[0]['model']}, prompt {ADJ[0]['prompt_version']}).\n")

print(f"accept a MATCH at confidence >= {CFG.get('CDP_ACCEPT_CONFIDENCE')}; "
      f"send to a steward below {CFG.get('CDP_STEWARD_CONFIDENCE')}\n")

for r in ADJ:
    print("=" * 78)
    show_record(r, cols=["record_id_a", "record_id_b", "tier", "combined_score",
                         "verdict", "confidence", "decisive_evidence",
                         "contradiction", "rationale", "injection_detected",
                         "model", "prompt_version", "adjudicated_at"])

## 8 · The match

The accepted pairs become edges in a graph, the graph is resolved into connected
components, and each component becomes one person. Here is the customer's cluster:
every source record that ended up pointing at the same human.

In [ ]:
CLUSTER_SQL = """
SELECT pa.record_id, pr.source_system, pr.source_trust,
       pr.raw_name, pr.name_norm, pr.email_norm, pr.phone_e164,
       pr.account_number, pr.identity_strength
FROM `{ds}.person_assignment` AS pa
JOIN `{ds}.party_records`     AS pr USING (record_id)
WHERE pa.person_id = @pid
ORDER BY pr.source_trust DESC, pa.record_id
LIMIT 30
"""
CLUSTER = q(CLUSTER_SQL.replace("@pid", lit(PERSON_ID)))
print(f"{PERSON_ID} — {len(CLUSTER)} source records, "
      f"{len({r['source_system'] for r in CLUSTER})} systems\n")
show(CLUSTER, width=30)

### The surviving record

Those records disagree, so something has to choose. Survivorship picks a winner per
field, weighted by how much each source is trusted and how recent the assertion is.
This is the row an agent, a campaign or a service desk would be handed.

In [ ]:
GOLDEN = q("SELECT * FROM `{ds}.golden_person` WHERE person_id = "
           + lit(PERSON_ID), max_rows=1)
if GOLDEN:
    show_record(GOLDEN[0],
                cols=["person_id", "full_name", "forename", "surname",
                      "address", "city", "postcode", "email", "phone", "dob",
                      "account_number", "populated_fields", "contested_fields",
                      "best_source_trust", "most_recent_assertion"])
else:
    print("No golden_person row — stage 80 has not been run.")

### Which source won each field, and what it beat

`was_contested` is the honest column. Where it is true, the sources genuinely
disagreed and the pipeline made a choice — `provenance` records what it chose over,
and `won_from_source` records who won. That trail is what makes the merged record
defensible rather than merely tidy.

In [ ]:
SURV_SQL = """
SELECT field, surviving_value, won_from_source, source_trust,
       won_from_record_id, asserting_sources, distinct_values,
       was_contested, provenance
FROM `{ds}.field_survivorship`
WHERE person_id = @pid
ORDER BY was_contested DESC, field
LIMIT 30
"""
SURV = q(SURV_SQL.replace("@pid", lit(PERSON_ID)))
show(SURV, cols=["field", "surviving_value", "won_from_source", "source_trust",
                 "asserting_sources", "distinct_values", "was_contested"],
     width=34)

print("\ncontested fields, in full")
_contested = [r for r in SURV if str(r.get("was_contested")).lower() == "true"]
if not _contested:
    print("  none — every source that had an opinion agreed")
for r in _contested:
    print(f"\n  {r['field']}  ->  {r['surviving_value']}")
    print(f"    {r['provenance']}")

print(f"\nOne person. {len(CLUSTER)} records in, 1 record out, "
      f"{len(_contested)} field(s) where a choice had to be made.")

## Scorecard for this run

The customer above is one story. These are the numbers for the whole corpus, marked
against ground truth the pipeline is never allowed to read.

Read `clusters_containing_multiple_people` before anything else: an over-merge and an
under-merge are not interchangeable, because one of them is a privacy incident.
`pct_of_pairs_using_an_llm` is the cost control — if that is not small, the tiering
above it needs attention before the budget does.

In [ ]:
SCORE = q("SELECT * FROM `{ds}.v_scorecard`", max_rows=1)
if SCORE:
    show_record(SCORE[0], cols=[
        "records", "true_people", "predicted_people", "people_count_ratio",
        "clusters_containing_multiple_people", "people_split_across_clusters",
        "pairwise_precision", "pairwise_recall", "pairwise_f1", "tp", "fp", "fn",
        "pairs_evaluated", "pairs_sent_to_llm", "pct_of_pairs_using_an_llm",
        "pairs_queued_for_a_human", "pct_of_decisions_deferred",
        "tau_high", "tau_low", "accept_confidence",
        "adjudicator_model", "prompt_version", "embedding_endpoint",
        "generator_seed", "scored_at"])
else:
    print("No scorecard — stage 95 has not been run.")